# Thesis imputation

1. 'Other' race group excluded
2. Primary metric switched to AUPRC
3. Statistical analysis, bootstrap permutation tests + FDR correction
4. LSTM missingness imputation experiments with counterfactual fairness

---
## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import itertools
from pathlib import Path
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve
from sklearn.model_selection import StratifiedKFold
import torch
import torch.nn as nn
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

OUT_DIR = Path('/home/ino/thesis_outputs')
EXT_DIR = OUT_DIR / 'extension'
EXT_DIR.mkdir(exist_ok=True)

VITAL_NAMES  = ['heart_rate', 'resp_rate', 'spo2', 'temperature', 'map']
KEEP_RACES   = ['White', 'Black', 'Hispanic', 'Asian']
MODEL_NAMES  = ['XGBoost', 'Deep NN', 'LSTM']
N_FOLDS      = 5
RANDOM_STATE = 42
N_BOOTSTRAP  = 2000

COLORS = {'XGBoost': 'steelblue', 'Deep NN': 'seagreen', 'LSTM': 'darkorange'}
RACE_COLORS = {'White': '#4878CF', 'Black': '#D65F5F',
               'Hispanic': '#6ACC65', 'Asian': '#B47CC7'}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Output: {EXT_DIR}')

---
## 2. Load & align all data


In [ ]:
# Load the ground-truth CV ordering saved by fix_alignment.py
cv_stay_ids = np.load(OUT_DIR / 'cv_stay_ids.npy')
print(f'CV stay_ids loaded: {len(cv_stay_ids):,}')

# Load cohort and reindex to exact CV order
cohort_full = pd.read_csv(OUT_DIR / 'cohort.csv')
cohort_full = cohort_full.set_index('stay_id')
cohort_cv   = cohort_full.loc[cv_stay_ids].reset_index()
print(f'cohort_cv rows: {len(cohort_cv):,}')
print(cohort_cv['race_group'].value_counts().to_string())

# Load OOF probability arrays
oof_probs_full = {
    'XGBoost': np.load(OUT_DIR / 'oof_probs_XGBoost.npy'),
    'Deep NN': np.load(OUT_DIR / 'oof_probs_Deep_NN.npy'),
    'LSTM':    np.load(OUT_DIR / 'oof_probs_LSTM.npy'),
}
print(f'OOF array length: {len(next(iter(oof_probs_full.values()))):,}')

# Load TS arrays and reindex to CV order
cohort_full_reset = pd.read_csv(OUT_DIR / 'cohort.csv').reset_index(drop=True)
stay_id_to_ts_idx = {sid: i for i, sid in enumerate(cohort_full_reset['stay_id'].values)}

X_ts_full  = np.load(OUT_DIR / 'X_ts.npy')
M_ts_full  = np.load(OUT_DIR / 'M_ts.npy')
print(f'X_ts_full: {X_ts_full.shape}')

ts_indices = np.array([stay_id_to_ts_idx[sid] for sid in cv_stay_ids])
X_ts_cv    = X_ts_full[ts_indices]
M_ts_cv    = M_ts_full[ts_indices]
print(f'X_ts_cv: {X_ts_cv.shape}')

# Filter out Other
keep_mask = cohort_cv['race_group'].isin(KEEP_RACES).values
cohort    = cohort_cv[keep_mask].reset_index(drop=True)
kept_idx  = np.where(keep_mask)[0]

X_ts      = X_ts_cv[kept_idx]
M_ts      = M_ts_cv[kept_idx]
oof_probs = {m: oof_probs_full[m][kept_idx] for m in MODEL_NAMES}

y    = cohort['hospital_expire_flag'].values.astype(np.int32)
demo = cohort[['race_group', 'gender', 'age_group']].reset_index(drop=True)

print(f'\nFinal cohort (Other excluded): {len(cohort):,}')
print(cohort['race_group'].value_counts().to_string())
print(f'Mortality rate: {y.mean():.1%}')
print(f'X_ts shape: {X_ts.shape}')
print('\nOOF performance (sanity check — should match original ~0.42 AUPRC):')
for m in MODEL_NAMES:
    print(f'  {m:<12}  AUPRC={average_precision_score(y, oof_probs[m]):.4f}  '
          f'AUROC={roc_auc_score(y, oof_probs[m]):.4f}')

---
## 3. AUPRC by Race Group

In [ ]:
rows = []
for m in MODEL_NAMES:
    probs = oof_probs[m]
    rows.append({'model': m, 'group': 'Overall',
                 'auprc': average_precision_score(y, probs),
                 'n': len(y), 'n_pos': int(y.sum())})
    for g in KEEP_RACES:
        mask = (demo['race_group'] == g).values
        yg, pg = y[mask], probs[mask]
        if yg.sum() < 5:
            continue
        rows.append({'model': m, 'group': g,
                     'auprc': average_precision_score(yg, pg),
                     'n': int(mask.sum()), 'n_pos': int(yg.sum())})

auprc_df = pd.DataFrame(rows)
print('AUPRC by race group (OOF):')
pivot = auprc_df[auprc_df['group'] != 'Overall'].pivot(
    index='group', columns='model', values='auprc').round(4)
print(pivot.to_string())
auprc_df.to_csv(EXT_DIR / 'auprc_by_group.csv', index=False)

In [ ]:
fig, axes = plt.subplots(1, len(KEEP_RACES), figsize=(5 * len(KEEP_RACES), 4))
for ax, g in zip(axes, KEEP_RACES):
    mask = (demo['race_group'] == g).values
    yg   = y[mask]
    ax.axhline(yg.mean(), color='gray', linestyle='--', alpha=0.5,
               label=f'Baseline ({yg.mean():.2f})')
    for m in MODEL_NAMES:
        pg = oof_probs[m][mask]
        prec, rec, _ = precision_recall_curve(yg, pg)
        ap = average_precision_score(yg, pg)
        ax.plot(rec, prec, color=COLORS[m], linewidth=1.8,
                label=f'{m} ({ap:.3f})')
    ax.set_title(f'{g}  (n={mask.sum():,})')
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.set_xlim(0,1); ax.set_ylim(0,1); ax.legend(fontsize=8)
plt.suptitle('Precision-recall curves by race group — out-of-fold', fontsize=13)
plt.tight_layout()
plt.savefig(EXT_DIR / 'fig_pr_curves_by_race.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_pr_curves_by_race.png')

In [ ]:
x = np.arange(len(KEEP_RACES))
width = 0.25
fig, ax = plt.subplots(figsize=(10, 5))
for offset, m in enumerate(MODEL_NAMES):
    vals = [auprc_df[(auprc_df['model']==m)&(auprc_df['group']==g)]['auprc'].values[0]
            for g in KEEP_RACES]
    ax.bar(x + offset*width, vals, width, label=m,
           color=list(COLORS.values())[offset], edgecolor='white', alpha=0.85)
for m in MODEL_NAMES:
    ov = auprc_df[(auprc_df['model']==m)&(auprc_df['group']=='Overall')]['auprc'].values[0]
    ax.axhline(ov, color=COLORS[m], linestyle='--', linewidth=1, alpha=0.6)
ax.set_xticks(x + width)
ax.set_xticklabels(KEEP_RACES)
ax.set_ylabel('AUPRC')
ax.set_title('AUPRC by race group — out-of-fold (Other excluded)')
ax.legend()
plt.tight_layout()
plt.savefig(EXT_DIR / 'fig_auprc_by_race.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_auprc_by_race.png')

---
## 4. Statistical Analysis

In [ ]:
def bootstrap_auprc_ci(y_true, probs, n_boot=N_BOOTSTRAP, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    scores = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        ys, ps = y_true[idx], probs[idx]
        if ys.sum() == 0 or ys.sum() == n:
            continue
        scores.append(average_precision_score(ys, ps))
    scores = np.array(scores)
    return scores.mean(), np.percentile(scores, 2.5), np.percentile(scores, 97.5)

print('4a. Bootstrap 95% CIs for AUPRC (2000 resamples)...')
print('=' * 65)
ci_records = []
for m in MODEL_NAMES:
    probs = oof_probs[m]
    print(f'\n{m}:')
    mn, lo, hi = bootstrap_auprc_ci(y, probs)
    print(f'  {"Overall":<12}  {mn:.4f}  95% CI [{lo:.4f}, {hi:.4f}]')
    ci_records.append({'model': m, 'group': 'Overall',
                       'auprc': mn, 'ci_lower': lo, 'ci_upper': hi})
    for g in KEEP_RACES:
        mask = (demo['race_group'] == g).values
        yg, pg = y[mask], probs[mask]
        if yg.sum() < 10:
            continue
        mn, lo, hi = bootstrap_auprc_ci(yg, pg)
        print(f'  {g:<12}  {mn:.4f}  95% CI [{lo:.4f}, {hi:.4f}]')
        ci_records.append({'model': m, 'group': g,
                           'auprc': mn, 'ci_lower': lo, 'ci_upper': hi})

ci_df = pd.DataFrame(ci_records)
ci_df.to_csv(EXT_DIR / 'auprc_bootstrap_ci.csv', index=False)
print('\nSaved auprc_bootstrap_ci.csv')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, m in zip(axes, MODEL_NAMES):
    sub = ci_df[(ci_df['model']==m) & (ci_df['group']!='Overall')].sort_values('auprc')
    y_pos = np.arange(len(sub))
    ax.barh(y_pos, sub['auprc'], color=COLORS[m], alpha=0.75, height=0.5)
    ax.errorbar(sub['auprc'], y_pos,
                xerr=[sub['auprc']-sub['ci_lower'], sub['ci_upper']-sub['auprc']],
                fmt='none', color='black', capsize=5, linewidth=1.8)
    overall = ci_df[(ci_df['model']==m)&(ci_df['group']=='Overall')]['auprc'].values[0]
    ax.axvline(overall, color='crimson', linestyle='--', linewidth=1.5,
               label=f'Overall ({overall:.3f})')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(sub['group'])
    ax.set_xlabel('AUPRC  (95% bootstrap CI)')
    ax.set_title(m)
    ax.legend(fontsize=9)
plt.suptitle('AUPRC with 95% bootstrap CIs by race group — Other excluded', fontsize=13)
plt.tight_layout()
plt.savefig(EXT_DIR / 'fig_auprc_bootstrap_ci.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_auprc_bootstrap_ci.png')

In [ ]:
def permutation_test_auprc(y_true, probs, mask_a, mask_b,
                            n_perm=N_BOOTSTRAP, seed=0):
    rng    = np.random.default_rng(seed)
    ya, pa = y_true[mask_a], probs[mask_a]
    yb, pb = y_true[mask_b], probs[mask_b]
    if ya.sum() == 0 or yb.sum() == 0:
        return np.nan, np.nan, np.nan, np.nan
    obs_a    = average_precision_score(ya, pa)
    obs_b    = average_precision_score(yb, pb)
    obs_diff = obs_a - obs_b
    y_pool   = np.concatenate([ya, yb])
    p_pool   = np.concatenate([pa, pb])
    na       = len(ya)
    perm_diffs = []
    for _ in range(n_perm):
        idx = rng.permutation(len(y_pool))
        ya_p, pa_p = y_pool[idx[:na]], p_pool[idx[:na]]
        yb_p, pb_p = y_pool[idx[na:]], p_pool[idx[na:]]
        if ya_p.sum() == 0 or yb_p.sum() == 0:
            continue
        perm_diffs.append(
            average_precision_score(ya_p, pa_p) -
            average_precision_score(yb_p, pb_p)
        )
    perm_diffs = np.array(perm_diffs)
    p_val = np.mean(np.abs(perm_diffs) >= np.abs(obs_diff))
    return obs_diff, obs_a, obs_b, p_val

print('4b. Permutation test: Black vs White AUPRC')
print('=' * 65)
perm_records = []
for m in MODEL_NAMES:
    probs  = oof_probs[m]
    mask_b = (demo['race_group'] == 'Black').values
    mask_w = (demo['race_group'] == 'White').values
    diff, ap_b, ap_w, pval = permutation_test_auprc(y, probs, mask_b, mask_w)
    sig = '(significant)' if pval < 0.05 else '(not significant)'
    print(f'  {m:<12}  Black={ap_b:.4f}  White={ap_w:.4f}  '
          f'Diff={diff:+.4f}  p={pval:.4f}  {sig}')
    perm_records.append({'model': m, 'group_a': 'Black', 'group_b': 'White',
                         'auprc_a': ap_b, 'auprc_b': ap_w,
                         'diff': diff, 'p_value': pval})

In [ ]:
print('4c. All pairwise permutation tests with FDR correction...')
pairwise_records = []
for m in MODEL_NAMES:
    probs = oof_probs[m]
    for g1, g2 in itertools.combinations(KEEP_RACES, 2):
        mask1 = (demo['race_group'] == g1).values
        mask2 = (demo['race_group'] == g2).values
        diff, ap1, ap2, pval = permutation_test_auprc(y, probs, mask1, mask2)
        pairwise_records.append({'model': m, 'group_a': g1, 'group_b': g2,
                                 'auprc_a': ap1, 'auprc_b': ap2,
                                 'diff': diff, 'p_raw': pval})

pw_df = pd.DataFrame(pairwise_records)
fdr_parts = []
for m in MODEL_NAMES:
    sub = pw_df[pw_df['model']==m].copy()
    reject, p_corr, _, _ = multipletests(sub['p_raw'].values, method='fdr_bh')
    sub['p_fdr'] = p_corr
    sub['significant'] = reject
    fdr_parts.append(sub)
pw_df = pd.concat(fdr_parts).reset_index(drop=True)
pw_df.to_csv(EXT_DIR / 'pairwise_permutation_tests.csv', index=False)
print(pw_df[['model','group_a','group_b','auprc_a','auprc_b',
             'diff','p_raw','p_fdr','significant']].to_string(index=False))
print('\nSaved pairwise_permutation_tests.csv')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, m in zip(axes, MODEL_NAMES):
    sub = pw_df[pw_df['model']==m].reset_index(drop=True)
    bar_colors = ['#c0392b' if s else '#bdc3c7' for s in sub['significant']]
    ax.barh(range(len(sub)), sub['diff'], color=bar_colors,
            edgecolor='white', height=0.6)
    ax.axvline(0, color='black', linewidth=0.8)
    for i, (d, p, s) in enumerate(zip(sub['diff'], sub['p_fdr'], sub['significant'])):
        label = f'p={p:.3f}{"*" if s else ""}'
        ax.text(d + (0.001 if d >= 0 else -0.001), i, label,
                va='center', ha='left' if d >= 0 else 'right', fontsize=8)
    ax.set_yticks(range(len(sub)))
    ax.set_yticklabels([f'{r["group_a"]} vs {r["group_b"]}'
                        for _, r in sub.iterrows()], fontsize=9)
    ax.set_xlabel('AUPRC difference (A - B)')
    ax.set_title(f'{m}  (red = FDR significant)')
plt.suptitle('Pairwise AUPRC differences — permutation test + FDR correction', fontsize=12)
plt.tight_layout()
plt.savefig(EXT_DIR / 'fig_pairwise_perm_tests.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_pairwise_perm_tests.png')

In [ ]:
def paired_bootstrap_auprc(y_true, probs_a, probs_b,
                            n_boot=N_BOOTSTRAP, seed=1):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    diffs = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        ys  = y_true[idx]
        if ys.sum() == 0 or ys.sum() == n:
            continue
        diffs.append(
            average_precision_score(ys, probs_a[idx]) -
            average_precision_score(ys, probs_b[idx])
        )
    diffs = np.array(diffs)
    p_val = 2 * min(np.mean(diffs <= 0), np.mean(diffs >= 0))
    return diffs.mean(), diffs.std(), np.percentile(diffs, 2.5), np.percentile(diffs, 97.5), p_val

print('4d. Paired bootstrap model comparison on AUPRC')
print('=' * 65)
model_comp_records = []
for ma, mb in itertools.combinations(MODEL_NAMES, 2):
    mn, sd, lo, hi, pval = paired_bootstrap_auprc(y, oof_probs[ma], oof_probs[mb])
    print(f'  {ma} vs {mb}:  mean diff={mn:+.4f}  '
          f'CI [{lo:+.4f}, {hi:+.4f}]  p={pval:.4f}')
    model_comp_records.append({'model_a': ma, 'model_b': mb,
                               'mean_diff': mn, 'ci_lower': lo,
                               'ci_upper': hi, 'p_value': pval})
pd.DataFrame(model_comp_records).to_csv(
    EXT_DIR / 'model_comparison_auprc.csv', index=False)
print('\nSaved model_comparison_auprc.csv')

---
## 5. LSTM Missingness Imputation Experiments

In [ ]:
class MortalityLSTM(nn.Module):
    def __init__(self, input_size=10, hidden_size=128, num_layers=3, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size, hidden_size=hidden_size,
            num_layers=num_layers, batch_first=True,
            dropout=dropout, bidirectional=True
        )
        self.layer_norm = nn.LayerNorm(hidden_size * 2)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size*2, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64),            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        last = self.layer_norm(out[:, -1, :])
        return self.classifier(self.dropout(last)).squeeze(1)

def lstm_predict(model, X_input, batch_size=1024):
    model.eval()
    probs = []
    with torch.no_grad():
        for s in range(0, len(X_input), batch_size):
            xb = torch.tensor(X_input[s:s+batch_size],
                              dtype=torch.float32).to(device)
            probs.append(torch.sigmoid(model(xb)).cpu().numpy())
    return np.concatenate(probs)

def normalise_ts(X_input, train_idx):
    ts_mean = X_ts[train_idx].mean(axis=(0,1), keepdims=True)
    ts_std  = X_ts[train_idx].std(axis=(0,1),  keepdims=True) + 1e-8
    return (X_input - ts_mean) / ts_std

def build_lstm_input(X_norm, M):
    return np.concatenate([X_norm, M], axis=2).astype(np.float32)

print('LSTM helpers defined.')

In [ ]:
strat_label = (demo['race_group'].astype(str) + '_' +
               cohort['hospital_expire_flag'].astype(str)).values
skf         = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
fold_splits  = list(skf.split(np.zeros(len(cohort)), strat_label))
race_arr     = demo['race_group'].values
print(f'Fold sizes: {[len(t) for _, t in fold_splits]}')

In [ ]:
white_train_dist = {}
black_obs_rate   = {}

for fold, (train_idx, _) in enumerate(fold_splits):
    white_idx = train_idx[race_arr[train_idx] == 'White']
    black_idx = train_idx[race_arr[train_idx] == 'Black']
    X_white   = X_ts[white_idx]
    M_white   = M_ts[white_idx]
    M_black   = M_ts[black_idx]
    fold_dist = {}
    fold_rate = {}
    for v in range(len(VITAL_NAMES)):
        fold_dist[v] = {}
        fold_rate[v] = {}
        for h in range(24):
            obs_mask        = M_white[:, h, v] == 1
            fold_dist[v][h] = X_white[:, h, v][obs_mask] if obs_mask.any() else None
            fold_rate[v][h] = float(M_black[:, h, v].mean())
    white_train_dist[fold] = fold_dist
    black_obs_rate[fold]   = fold_rate

print('Per-fold distributions built.')
print('Mean Black observation rate per vital:')
for v, vname in enumerate(VITAL_NAMES):
    rates = [black_obs_rate[f][v][h] for f in range(N_FOLDS) for h in range(24)]
    print(f'  {vname:<14}  {np.mean(rates):.3f}')

In [ ]:
rng          = np.random.default_rng(42)
oof_baseline = np.zeros(len(cohort))
oof_exp_a    = np.zeros(len(cohort))
oof_exp_b    = np.zeros(len(cohort))

print('Running LSTM imputation experiments across 5 folds...')
print('=' * 60)

for fold, (train_idx, test_idx) in enumerate(fold_splits):
    print(f'\nFold {fold+1}/5')
    lstm_model = MortalityLSTM(input_size=10, hidden_size=128,
                               num_layers=3, dropout=0.3)
    state = torch.load(OUT_DIR / f'lstm_fold{fold+1}.pt', map_location=device)
    lstm_model.load_state_dict(state)
    lstm_model = lstm_model.to(device)

    X_te      = X_ts[test_idx].copy()
    M_te      = M_ts[test_idx].copy()
    X_te_norm = normalise_ts(X_te, train_idx)
    oof_baseline[test_idx] = lstm_predict(
        lstm_model, build_lstm_input(X_te_norm, M_te))

    race_te   = race_arr[test_idx]
    black_loc = np.where(race_te == 'Black')[0]
    white_loc = np.where(race_te == 'White')[0]
    fdist     = white_train_dist[fold]
    brate     = black_obs_rate[fold]

    # Experiment A
    X_te_a = X_te.copy()
    M_te_a = M_te.copy()
    for li in black_loc:
        for v in range(len(VITAL_NAMES)):
            for h in range(24):
                if M_te_a[li, h, v] == 0:
                    vals   = fdist[v][h]
                    w_rate = float(M_ts[train_idx][race_arr[train_idx]=='White', h, v].mean())
                    if vals is not None and len(vals) > 0 and rng.random() < w_rate:
                        X_te_a[li, h, v] = rng.choice(vals)
                        M_te_a[li, h, v] = 1.0
    X_te_a_norm        = normalise_ts(X_te_a, train_idx)
    oof_exp_a[test_idx] = lstm_predict(
        lstm_model, build_lstm_input(X_te_a_norm, M_te_a))

    # Experiment B
    X_te_b = X_te.copy()
    M_te_b = M_te.copy()
    for li in white_loc:
        for v in range(len(VITAL_NAMES)):
            for h in range(24):
                if M_te_b[li, h, v] == 1 and rng.random() > brate[v][h]:
                    M_te_b[li, h, v] = 0.0
                    X_te_b[li, h, v] = float(np.median(X_ts[train_idx][:, h, v]))
    X_te_b_norm        = normalise_ts(X_te_b, train_idx)
    oof_exp_b[test_idx] = lstm_predict(
        lstm_model, build_lstm_input(X_te_b_norm, M_te_b))

    y_te = y[test_idx]
    if y_te[race_te=='Black'].sum() > 0:
        bl_b = average_precision_score(y_te[race_te=='Black'],
                                       oof_baseline[test_idx][race_te=='Black'])
        bl_a = average_precision_score(y_te[race_te=='Black'],
                                       oof_exp_a[test_idx][race_te=='Black'])
        print(f'  Black  baseline={bl_b:.4f}  ExpA={bl_a:.4f}  delta={bl_a-bl_b:+.4f}')
    if y_te[race_te=='White'].sum() > 0:
        wh_b  = average_precision_score(y_te[race_te=='White'],
                                        oof_baseline[test_idx][race_te=='White'])
        wh_b2 = average_precision_score(y_te[race_te=='White'],
                                        oof_exp_b[test_idx][race_te=='White'])
        print(f'  White  baseline={wh_b:.4f}  ExpB={wh_b2:.4f}  delta={wh_b2-wh_b:+.4f}')

    del lstm_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print('\nAll folds complete.')

In [ ]:
mask_black = (demo['race_group'] == 'Black').values
mask_white = (demo['race_group'] == 'White').values
y_black    = y[mask_black]
y_white    = y[mask_white]

results_exp = {
    'Black — Baseline':               average_precision_score(y_black, oof_baseline[mask_black]),
    'Black — Exp A (White patterns)': average_precision_score(y_black, oof_exp_a[mask_black]),
    'White — Baseline':               average_precision_score(y_white, oof_baseline[mask_white]),
    'White — Exp B (Black patterns)': average_precision_score(y_white, oof_exp_b[mask_white]),
}
print('AUPRC summary:')
for k, v in results_exp.items():
    print(f'  {k:<45}  {v:.4f}')

delta_a = results_exp['Black — Exp A (White patterns)'] - results_exp['Black — Baseline']
delta_b = results_exp['White — Exp B (Black patterns)'] - results_exp['White — Baseline']
print(f'\n  Delta A: {delta_a:+.4f}')
print(f'  Delta B: {delta_b:+.4f}')

In [ ]:
def bootstrap_delta_ci(y_true, probs_base, probs_exp,
                        n_boot=N_BOOTSTRAP, seed=99):
    rng    = np.random.default_rng(seed)
    n      = len(y_true)
    deltas = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        ys  = y_true[idx]
        if ys.sum() == 0 or ys.sum() == n:
            continue
        deltas.append(
            average_precision_score(ys, probs_exp[idx]) -
            average_precision_score(ys, probs_base[idx])
        )
    deltas = np.array(deltas)
    p_val  = 2 * min(np.mean(deltas <= 0), np.mean(deltas >= 0))
    return deltas.mean(), np.percentile(deltas, 2.5), np.percentile(deltas, 97.5), p_val

mn_a, lo_a, hi_a, p_a = bootstrap_delta_ci(
    y_black, oof_baseline[mask_black], oof_exp_a[mask_black])
mn_b, lo_b, hi_b, p_b = bootstrap_delta_ci(
    y_white, oof_baseline[mask_white], oof_exp_b[mask_white])

print(f'Exp A delta: {mn_a:+.4f}  95% CI [{lo_a:+.4f}, {hi_a:+.4f}]  p={p_a:.4f}')
print(f'Exp B delta: {mn_b:+.4f}  95% CI [{lo_b:+.4f}, {hi_b:+.4f}]  p={p_b:.4f}')

exp_summary = pd.DataFrame([
    {'experiment': 'A — Black with White patterns', 'group': 'Black',
     'auprc_baseline': results_exp['Black — Baseline'],
     'auprc_experiment': results_exp['Black — Exp A (White patterns)'],
     'delta': mn_a, 'ci_lower': lo_a, 'ci_upper': hi_a, 'p_value': p_a},
    {'experiment': 'B — White with Black patterns', 'group': 'White',
     'auprc_baseline': results_exp['White — Baseline'],
     'auprc_experiment': results_exp['White — Exp B (Black patterns)'],
     'delta': mn_b, 'ci_lower': lo_b, 'ci_upper': hi_b, 'p_value': p_b},
])
exp_summary.to_csv(EXT_DIR / 'imputation_experiment_summary.csv', index=False)
print('Saved imputation_experiment_summary.csv')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax_i, (group, key_base, key_exp, mn, lo, hi, p) in enumerate([
    ('Black','Black — Baseline','Black — Exp A (White patterns)', mn_a, lo_a, hi_a, p_a),
    ('White','White — Baseline','White — Exp B (Black patterns)', mn_b, lo_b, hi_b, p_b),
]):
    ax = axes[ax_i]
    vals    = [results_exp[key_base], results_exp[key_exp]]
    labels  = ['Baseline', 'Experiment']
    bcolors = ['#D65F5F','#4878CF'] if group=='Black' else ['#4878CF','#D65F5F']
    bars = ax.bar(labels, vals, color=bcolors, edgecolor='white', width=0.5, alpha=0.85)
    ax.errorbar(1, vals[1], yerr=[[mn-lo],[hi-mn]],
                fmt='none', color='black', capsize=6, linewidth=2)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, val+0.002,
                f'{val:.4f}', ha='center', fontsize=10, fontweight='bold')
    title = ('Experiment A — Black patients\n(given White measurement patterns)'
             if group=='Black' else
             'Experiment B — White patients\n(given Black measurement patterns)')
    ax.set_title(title); ax.set_ylabel('AUPRC')
    ax.text(0.5, 0.05, f'delta={mn:+.4f}  p={p:.3f}',
            ha='center', transform=ax.transAxes, fontsize=10,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))
    ax.set_ylim(0, max(vals)*1.15)
plt.suptitle('LSTM counterfactual missingness experiments', fontsize=12)
plt.tight_layout()
plt.savefig(EXT_DIR / 'fig_imputation_experiments.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_imputation_experiments.png')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax_i, (group, y_grp, p_base, p_exp, c_base, c_exp, label_exp) in enumerate([
    ('Black', y_black, oof_baseline[mask_black], oof_exp_a[mask_black],
     '#D65F5F','#4878CF','Black + White patterns'),
    ('White', y_white, oof_baseline[mask_white], oof_exp_b[mask_white],
     '#4878CF','#D65F5F','White + Black patterns'),
]):
    ax = axes[ax_i]
    ap_base = average_precision_score(y_grp, p_base)
    ap_exp  = average_precision_score(y_grp, p_exp)
    prec_b, rec_b, _ = precision_recall_curve(y_grp, p_base)
    prec_e, rec_e, _ = precision_recall_curve(y_grp, p_exp)
    ax.plot(rec_b, prec_b, color=c_base, linewidth=2,
            label=f'{group} baseline (AUPRC={ap_base:.3f})')
    ax.plot(rec_e, prec_e, color=c_exp, linewidth=2, linestyle='--',
            label=f'{label_exp} (AUPRC={ap_exp:.3f})')
    ax.axhline(y_grp.mean(), color='gray', linestyle=':', alpha=0.6,
               label=f'Prevalence ({y_grp.mean():.2f})')
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.set_title(f'Experiment {"A" if group=="Black" else "B"} — {group} patients')
    ax.legend(fontsize=9); ax.set_xlim(0,1); ax.set_ylim(0,1)
plt.suptitle('PR curves — baseline vs counterfactual imputation', fontsize=13)
plt.tight_layout()
plt.savefig(EXT_DIR / 'fig_pr_curves_experiments.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_pr_curves_experiments.png')

---
## 6. Summary

In [ ]:
print('=' * 65)
print('EXTENSION ANALYSIS SUMMARY')
print('=' * 65)
print(f'\nCohort: {len(cohort):,}  |  Mortality: {y.mean():.1%}')
print(cohort['race_group'].value_counts().to_string())
print('\n── AUPRC by race group ──')
pivot = auprc_df[auprc_df['group']!='Overall'].pivot(
    index='group', columns='model', values='auprc').round(4)
print(pivot.to_string())
print('\n── AUPRC disparity gap (max - min) ──')
for m in MODEL_NAMES:
    sub = auprc_df[(auprc_df['model']==m) & (auprc_df['group']!='Overall')]
    print(f'  {m:<12}  {sub["auprc"].max()-sub["auprc"].min():.4f}')
print('\n── Black vs White permutation test ──')
for r in perm_records:
    sig = '(significant)' if r['p_value'] < 0.05 else '(not significant)'
    print(f'  {r["model"]:<12}  diff={r["diff"]:+.4f}  p={r["p_value"]:.4f}  {sig}')
print('\n── Imputation experiments ──')
print(exp_summary[['experiment','group','auprc_baseline',
                   'auprc_experiment','delta','p_value']].to_string(index=False))
print(f'\nOutputs saved to: {EXT_DIR}')
for f in sorted(EXT_DIR.glob('*')):
    print(f'  {f.name}')